<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: Agent Execution Logging </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**Purpose:**  
Demonstrate how to implement **Agent Execution Logging** in LangGraph-based agent systems — capturing execution traces, logging tool calls, tracking reasoning steps, and measuring cost & latency.

**Scenario:** Financial Research Agent  
A LangGraph agent that researches companies, analyzes financial data, and generates investment summaries — while a custom observability layer silently records every decision, tool invocation, and reasoning chain for debugging and auditing.

**Key Topics Covered:**
- Logging agent decisions and reasoning steps
- Recording tool calls with inputs / outputs
- Building structured execution traces
- Tracking latency per step and total cost estimation

---


## 1. Setup

In [1]:
# Run this cell first
!pip install -q langchain langchain-openai langgraph langchain-community --quiet

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Enter your OpenAI API key: ··········
Setup complete


In [2]:
import time
import uuid
import json
import logging
from datetime import datetime
from typing import Annotated, TypedDict, List, Dict, Any, Optional

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

# LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("LLM and libraries ready")


LLM and libraries ready


## 2. Observability Layer — Core Infrastructure

Before defining any tools or agents we build the **observability backbone**:

| Component | Purpose |
|---|---|
| `ExecutionTrace` | Structured record for a single agent run |
| `StepLog` | One logged event (tool call, decision, reasoning) |
| `ObservabilityLogger` | Singleton that stores all traces and prints reports |

This mirrors production systems like LangSmith or Weights & Biases, but runs entirely in-process so you can see exactly what is captured.


In [3]:
# Data structures

from dataclasses import dataclass, field

@dataclass
class StepLog:
    """A single logged step inside an agent run."""
    step_id:    str
    step_type:  str          # 'reasoning' | 'tool_call' | 'tool_result' | 'decision'
    timestamp:  str
    content:    Any
    latency_ms: float = 0.0
    metadata:   Dict[str, Any] = field(default_factory=dict)


@dataclass
class ExecutionTrace:
    """Full trace for one agent invocation."""
    trace_id:       str
    query:          str
    started_at:     str
    steps:          List[StepLog] = field(default_factory=list)
    finished_at:    Optional[str] = None
    total_ms:       float = 0.0
    estimated_cost: float = 0.0   # USD — rough estimate
    final_answer:   Optional[str] = None
    status:         str = "running"   # running | success | error


class ObservabilityLogger:
    """Central store for all execution traces."""

    def __init__(self):
        self.traces: Dict[str, ExecutionTrace] = {}
        self._current_trace_id: Optional[str] = None

    # Trace lifecycle

    def start_trace(self, query: str) -> str:
        trace_id = str(uuid.uuid4())[:8]
        self.traces[trace_id] = ExecutionTrace(
            trace_id=trace_id,
            query=query,
            started_at=datetime.utcnow().isoformat(),
        )
        self._current_trace_id = trace_id
        print(f"\n{'='*65}")
        print(f"    NEW TRACE STARTED  [{trace_id}]")
        print(f"  Query: {query[:80]}{'...' if len(query)>80 else ''}")
        print(f"{'='*65}")
        return trace_id

    def end_trace(self, trace_id: str, final_answer: str,
                  status: str = "success", total_tokens: int = 0):
        trace = self.traces[trace_id]
        trace.finished_at = datetime.utcnow().isoformat()
        start = datetime.fromisoformat(trace.started_at)
        end   = datetime.fromisoformat(trace.finished_at)
        trace.total_ms = (end - start).total_seconds() * 1000
        trace.final_answer = final_answer
        trace.status = status
        # gpt-4o-mini pricing: ~$0.15/1M input tokens (rough estimate)
        trace.estimated_cost = (total_tokens / 1_000_000) * 0.15
        print(f"\n{'='*65}")
        print(f"  TRACE COMPLETE  [{trace_id}]  status={status}")
        print(f"  Total time : {trace.total_ms:.0f} ms")
        print(f"  Est. cost  : ${trace.estimated_cost:.6f}")
        print(f"{'='*65}\n")

    #  Step logging

    def log_step(self, trace_id: str, step_type: str,
                 content: Any, latency_ms: float = 0.0,
                 metadata: Dict[str, Any] = None) -> StepLog:
        step = StepLog(
            step_id   = str(uuid.uuid4())[:6],
            step_type = step_type,
            timestamp = datetime.utcnow().isoformat(),
            content   = content,
            latency_ms= latency_ms,
            metadata  = metadata or {},
        )
        self.traces[trace_id].steps.append(step)

        # Pretty console output
        icons = {
            "reasoning":   "🧠",
            "tool_call":   "🔧",
            "tool_result": "📦",
            "decision":    "⚡",
        }
        icon = icons.get(step_type, "•")
        content_preview = str(content)[:120]
        print(f"  {icon} [{step_type.upper():12s}] {content_preview}")
        if latency_ms:
            print(f"              ⏱  {latency_ms:.0f} ms")
        return step

    # Reporting

    def print_full_report(self, trace_id: str):
        trace = self.traces[trace_id]
        print(f"\n{'#'*65}")
        print(f"  EXECUTION TRACE REPORT — [{trace_id}]")
        print(f"{'#'*65}")
        print(f"  Query        : {trace.query}")
        print(f"  Status       : {trace.status}")
        print(f"  Started      : {trace.started_at}")
        print(f"  Finished     : {trace.finished_at}")
        print(f"  Total time   : {trace.total_ms:.0f} ms")
        print(f"  Est. cost    : ${trace.estimated_cost:.6f}")
        print(f"  Steps logged : {len(trace.steps)}")
        print()

        for i, step in enumerate(trace.steps, 1):
            print(f"  Step {i:02d} | {step.step_type.upper():12s} | "
                  f"{step.latency_ms:6.0f} ms | {str(step.content)[:90]}")

        print()
        print(f"  Final Answer : {str(trace.final_answer)[:200]}")
        print(f"{'#'*65}\n")

    def print_summary_dashboard(self):
        """Print a high-level summary across all traces."""
        print(f"\n{'='*65}")
        print("  📊  OBSERVABILITY DASHBOARD — ALL TRACES")
        print(f"{'='*65}")
        print(f"  {'Trace':8s} {'Status':8s} {'Steps':6s} "
              f"{'Time(ms)':10s} {'Est.Cost':10s}  Query")
        print(f"  {'-'*60}")
        for t in self.traces.values():
            query_short = t.query[:35] + "..." if len(t.query) > 35 else t.query
            print(f"  {t.trace_id:8s} {t.status:8s} {len(t.steps):6d} "
                  f"{t.total_ms:10.0f} ${t.estimated_cost:.6f}  {query_short}")
        print(f"{'='*65}\n")


# Instantiate the global logger
obs_logger = ObservabilityLogger()

print("Observability logger initialised ")


Observability logger initialised 


## 3. Instrumented Financial Tools

Each tool is wrapped with **timing** and **structured logging** so every call appears in the execution trace automatically.  
Three tools simulate a real financial-research stack:

| Tool | What it does |
|---|---|
| `get_stock_price` | Returns current mock price + change |
| `search_financial_news` | Retrieves recent headlines for a ticker |
| `calculate_financial_metric` | Computes P/E ratio, market cap, etc. |


In [4]:
import random

# Mock financial data
MOCK_PRICES = {
    "AAPL": {"price": 189.50, "change_pct": +1.23},
    "MSFT": {"price": 415.20, "change_pct": -0.45},
    "NVDA": {"price": 875.60, "change_pct": +3.78},
    "TSLA": {"price": 172.30, "change_pct": -2.10},
    "GOOGL": {"price": 163.40, "change_pct": +0.89},
}

MOCK_NEWS = {
    "AAPL": [
        "Apple reports record iPhone 16 Pro sales in Q1 2025.",
        "Apple Vision Pro 2 launch confirmed for late 2025.",
        "Apple Services revenue grows 18% YoY, analysts bullish.",
    ],
    "MSFT": [
        "Microsoft Azure AI revenue surpasses $10B quarterly milestone.",
        "Copilot+ PC adoption accelerates across enterprise customers.",
        "Microsoft acquires AI safety startup Orbital Minds.",
    ],
    "NVDA": [
        "NVIDIA Blackwell GPU demand outpacing supply, CEO says.",
        "NVIDIA data-center revenue hits $22B in latest quarter.",
        "New CUDA 13 release boosts AI training throughput by 40%.",
    ],
    "TSLA": [
        "Tesla Full Self-Driving v13 rollout begins in Europe.",
        "Tesla Energy division sets quarterly profit record.",
        "Cybertruck deliveries increase 35% QoQ despite initial delays.",
    ],
    "GOOGL": [
        "Google Gemini Ultra 2 benchmarks surpass previous SOTA.",
        "Google Search ad revenue rebounds after AI Overview rollout.",
        "Waymo reaches 100,000 paid robotaxi rides per week.",
    ],
}

#  Tool definitions

@tool
def get_stock_price(ticker: str) -> str:
    """Get the current stock price and daily change for a given ticker symbol."""
    time.sleep(0.05)   # simulate network latency
    ticker = ticker.upper()
    if ticker in MOCK_PRICES:
        d = MOCK_PRICES[ticker]
        direction = "📈" if d["change_pct"] >= 0 else "📉"
        return (f"{ticker}: ${d['price']:.2f}  "
                f"{direction} {d['change_pct']:+.2f}% today")
    return f"Ticker '{ticker}' not found in dataset."


@tool
def search_financial_news(ticker: str) -> str:
    """Search for the latest financial news headlines for a given ticker symbol."""
    time.sleep(0.08)
    ticker = ticker.upper()
    headlines = MOCK_NEWS.get(ticker, [f"No recent news found for {ticker}."])
    return "\n".join(f"• {h}" for h in headlines)


@tool
def calculate_financial_metric(
    metric: str,
    price: float,
    eps: float = 1.0,
    shares_outstanding_M: float = 1000.0,
) -> str:
    """
    Calculate a financial metric.

    metric: 'pe_ratio' | 'market_cap' | 'earnings_yield'
    price: current stock price (USD)
    eps: earnings per share (USD)
    shares_outstanding_M: shares outstanding in millions
    """
    time.sleep(0.03)
    metric = metric.lower()
    if metric == "pe_ratio":
        if eps <= 0:
            return "Cannot calculate P/E: EPS must be positive."
        return f"P/E Ratio = {price / eps:.2f}x"
    elif metric == "market_cap":
        cap = price * shares_outstanding_M
        return f"Market Cap = ${cap:,.0f}M  (${cap/1000:.1f}B)"
    elif metric == "earnings_yield":
        if price <= 0:
            return "Cannot calculate: price must be positive."
        return f"Earnings Yield = {(eps / price) * 100:.2f}%"
    return f"Unknown metric '{metric}'. Choose: pe_ratio, market_cap, earnings_yield."


tools = [get_stock_price, search_financial_news, calculate_financial_metric]

print(f"{len(tools)} instrumented financial tools ready ")


3 instrumented financial tools ready ✔


## 4. LangGraph Agent with Observability Hooks

We build a **LangGraph ReAct-style agent** and inject observability hooks at each node:

```
START → [agent_node] → [tool_node] → [agent_node] → ... → END
                ↑               ↑
          logs reasoning    logs tool I/O
```

The agent state carries a `trace_id` so every log event is linked to the right trace.


In [5]:
from langgraph.graph.message import add_messages
from typing import Annotated

#  State
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    trace_id: str          # passed through the entire graph
    step_count: int
    total_tokens: int

#  Bind tools to LLM
llm_with_tools = llm.bind_tools(tools)

# Agent node
def agent_node(state: AgentState) -> AgentState:
    """Core reasoning node — calls the LLM and logs the decision."""
    trace_id   = state["trace_id"]
    step_count = state["step_count"] + 1
    t0 = time.perf_counter()

    response = llm_with_tools.invoke(state["messages"])
    latency  = (time.perf_counter() - t0) * 1000

    # Estimate token usage
    usage = getattr(response, "usage_metadata", None)
    new_tokens = 0
    if usage:
        new_tokens = usage.get("total_tokens", 0)
    total_tokens = state["total_tokens"] + new_tokens

    #  Log reasoning
    if response.content:
        obs_logger.log_step(
            trace_id   = trace_id,
            step_type  = "reasoning",
            content    = response.content,
            latency_ms = latency,
            metadata   = {"step": step_count, "tokens": new_tokens},
        )

    # Log tool call decisions
    if hasattr(response, "tool_calls") and response.tool_calls:
        for tc in response.tool_calls:
            obs_logger.log_step(
                trace_id  = trace_id,
                step_type = "decision",
                content   = f"Calling tool '{tc['name']}' with args {tc['args']}",
                metadata  = {"tool_name": tc["name"], "tool_args": tc["args"]},
            )

    return {
        "messages":    [response],
        "trace_id":    trace_id,
        "step_count":  step_count,
        "total_tokens": total_tokens,
    }


# Observable tool node
base_tool_node = ToolNode(tools)

def observable_tool_node(state: AgentState) -> AgentState:
    """Wraps ToolNode to log every tool invocation and its result."""
    trace_id = state["trace_id"]
    last_msg = state["messages"][-1]

    # Log each tool call before execution
    if hasattr(last_msg, "tool_calls"):
        for tc in last_msg.tool_calls:
            obs_logger.log_step(
                trace_id  = trace_id,
                step_type = "tool_call",
                content   = f"{tc['name']}({tc['args']})",
                metadata  = {"tool_name": tc["name"]},
            )

    t0 = time.perf_counter()
    result = base_tool_node.invoke(state)
    latency = (time.perf_counter() - t0) * 1000

    # Log results
    for msg in result.get("messages", []):
        if isinstance(msg, ToolMessage):
            obs_logger.log_step(
                trace_id   = trace_id,
                step_type  = "tool_result",
                content    = msg.content,
                latency_ms = latency,
                metadata   = {"tool_call_id": msg.tool_call_id},
            )

    return {**result, "trace_id": trace_id,
            "step_count": state["step_count"],
            "total_tokens": state["total_tokens"]}


# Routing
def should_continue(state: AgentState) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return END


#  Build the graph
graph = StateGraph(AgentState)
graph.add_node("agent",  agent_node)
graph.add_node("tools",  observable_tool_node)
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
graph.add_edge("tools", "agent")

agent = graph.compile()

print("LangGraph agent with observability hooks compiled ")


LangGraph agent with observability hooks compiled 


## 5. Helper — `run_observed_agent`

In [6]:
def run_observed_agent(query: str) -> str:
    """
    Convenience wrapper that:
      1. Opens a trace
      2. Runs the LangGraph agent
      3. Closes the trace with timing + cost
      4. Returns the final answer string
    """
    trace_id = obs_logger.start_trace(query)

    initial_state: AgentState = {
        "messages":     [HumanMessage(content=query)],
        "trace_id":     trace_id,
        "step_count":   0,
        "total_tokens": 0,
    }

    try:
        final_state = agent.invoke(initial_state)
        answer = final_state["messages"][-1].content
        status = "success"
        total_tokens = final_state.get("total_tokens", 0)
    except Exception as e:
        answer = f"Agent error: {e}"
        status = "error"
        total_tokens = 0

    obs_logger.end_trace(trace_id, answer,
                         status=status, total_tokens=total_tokens)

    return answer, trace_id

print("run_observed_agent helper ready ")


run_observed_agent helper ready 


## 6. Demo — Logging Agent Decisions

### Demo 1 — Single-tool query (simple reasoning)

Ask about one stock. We expect a **single `get_stock_price` tool call** followed by a short reasoning chain.


In [7]:
answer_1, trace_id_1 = run_observed_agent(
    "What is the current stock price of NVIDIA and how is it performing today?"
)

print("\nFINAL ANSWER:", answer_1)



    NEW TRACE STARTED  [536777d3]
  Query: What is the current stock price of NVIDIA and how is it performing today?


/tmp/ipykernel_1538/309376039.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  started_at=datetime.utcnow().isoformat(),
/tmp/ipykernel_1538/309376039.py:78: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat(),


  ⚡ [DECISION    ] Calling tool 'get_stock_price' with args {'ticker': 'NVDA'}
  🔧 [TOOL_CALL   ] get_stock_price({'ticker': 'NVDA'})
  📦 [TOOL_RESULT ] NVDA: $875.60  📈 +3.78% today
              ⏱  52 ms
  🧠 [REASONING   ] The current stock price of NVIDIA (NVDA) is $875.60, and it is performing well today with an increase of 3.78%.
              ⏱  1202 ms

  TRACE COMPLETE  [536777d3]  status=success
  Total time : 2108 ms
  Est. cost  : $0.000072


FINAL ANSWER: The current stock price of NVIDIA (NVDA) is $875.60, and it is performing well today with an increase of 3.78%.


/tmp/ipykernel_1538/309376039.py:78: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat(),
/tmp/ipykernel_1538/309376039.py:56: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  trace.finished_at = datetime.utcnow().isoformat()


### Demo 2 — Multi-tool reasoning (news + price + metric)

Ask a compound question that forces the agent to call **multiple tools** in sequence.


In [8]:
answer_2, trace_id_2 = run_observed_agent(
    "Research Apple (AAPL): get its current price, find the latest news headlines, "
    "then calculate its P/E ratio assuming EPS of $6.57."
)

print("\nFINAL ANSWER:", answer_2)


/tmp/ipykernel_1538/309376039.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  started_at=datetime.utcnow().isoformat(),



    NEW TRACE STARTED  [0d635cda]
  Query: Research Apple (AAPL): get its current price, find the latest news headlines, th...


/tmp/ipykernel_1538/309376039.py:78: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat(),


  ⚡ [DECISION    ] Calling tool 'get_stock_price' with args {'ticker': 'AAPL'}
  ⚡ [DECISION    ] Calling tool 'search_financial_news' with args {'ticker': 'AAPL'}
  ⚡ [DECISION    ] Calling tool 'calculate_financial_metric' with args {'metric': 'pe_ratio', 'price': 0, 'eps': 6.57}
  🔧 [TOOL_CALL   ] get_stock_price({'ticker': 'AAPL'})
  🔧 [TOOL_CALL   ] search_financial_news({'ticker': 'AAPL'})
  🔧 [TOOL_CALL   ] calculate_financial_metric({'metric': 'pe_ratio', 'price': 0, 'eps': 6.57})
  📦 [TOOL_RESULT ] AAPL: $189.50  📈 +1.23% today
              ⏱  87 ms
  📦 [TOOL_RESULT ] • Apple reports record iPhone 16 Pro sales in Q1 2025.
• Apple Vision Pro 2 launch confirmed for late 2025.
• Apple Serv
              ⏱  87 ms
  📦 [TOOL_RESULT ] P/E Ratio = 0.00x
              ⏱  87 ms
  🧠 [REASONING   ] Here's the latest information on Apple (AAPL):

- **Current Stock Price**: $189.50 (up 1.23% today)
  
- **Latest News H
              ⏱  2012 ms

  TRACE COMPLETE  [0d635cda]  status=success


/tmp/ipykernel_1538/309376039.py:78: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat(),
/tmp/ipykernel_1538/309376039.py:56: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  trace.finished_at = datetime.utcnow().isoformat()


### Demo 3 — Comparative analysis (multiple tickers)

Force the agent to retrieve data for **two companies** and synthesise a comparison.


In [9]:
answer_3, trace_id_3 = run_observed_agent(
    "Compare MSFT and GOOGL: fetch their prices, latest news, "
    "then calculate the market cap for each assuming 7,440M shares for MSFT "
    "and 12,200M shares for GOOGL."
)

print("\nFINAL ANSWER:", answer_3)


/tmp/ipykernel_1538/309376039.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  started_at=datetime.utcnow().isoformat(),



    NEW TRACE STARTED  [42e92c33]
  Query: Compare MSFT and GOOGL: fetch their prices, latest news, then calculate the mark...


/tmp/ipykernel_1538/309376039.py:78: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat(),


  ⚡ [DECISION    ] Calling tool 'get_stock_price' with args {'ticker': 'MSFT'}
  ⚡ [DECISION    ] Calling tool 'get_stock_price' with args {'ticker': 'GOOGL'}
  ⚡ [DECISION    ] Calling tool 'search_financial_news' with args {'ticker': 'MSFT'}
  ⚡ [DECISION    ] Calling tool 'search_financial_news' with args {'ticker': 'GOOGL'}
  ⚡ [DECISION    ] Calling tool 'calculate_financial_metric' with args {'metric': 'market_cap', 'price': 0, 'shares_outstanding_M': 7440}
  ⚡ [DECISION    ] Calling tool 'calculate_financial_metric' with args {'metric': 'market_cap', 'price': 0, 'shares_outstanding_M': 12200}
  🔧 [TOOL_CALL   ] get_stock_price({'ticker': 'MSFT'})
  🔧 [TOOL_CALL   ] get_stock_price({'ticker': 'GOOGL'})
  🔧 [TOOL_CALL   ] search_financial_news({'ticker': 'MSFT'})
  🔧 [TOOL_CALL   ] search_financial_news({'ticker': 'GOOGL'})
  🔧 [TOOL_CALL   ] calculate_financial_metric({'metric': 'market_cap', 'price': 0, 'shares_outstanding_M': 7440})
  🔧 [TOOL_CALL   ] calculate_financial_metric

/tmp/ipykernel_1538/309376039.py:78: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat(),
/tmp/ipykernel_1538/309376039.py:56: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  trace.finished_at = datetime.utcnow().isoformat()


## 7. Execution Trace Inspection

Print the full, step-by-step trace for any of the runs above.  
Each row shows: **step number | step type | latency | content preview**.


In [10]:
#  Detailed report for Demo 2 (most interesting — multi-tool)
print("\n>>> Full trace for Demo 2 (Apple multi-tool query):\n")
obs_logger.print_full_report(trace_id_2)



>>> Full trace for Demo 2 (Apple multi-tool query):


#################################################################
  EXECUTION TRACE REPORT — [0d635cda]
#################################################################
  Query        : Research Apple (AAPL): get its current price, find the latest news headlines, then calculate its P/E ratio assuming EPS of $6.57.
  Status       : success
  Started      : 2026-06-16T11:57:28.591725
  Finished     : 2026-06-16T11:57:32.076511
  Total time   : 3485 ms
  Est. cost    : $0.000120
  Steps logged : 10

  Step 01 | DECISION     |      0 ms | Calling tool 'get_stock_price' with args {'ticker': 'AAPL'}
  Step 02 | DECISION     |      0 ms | Calling tool 'search_financial_news' with args {'ticker': 'AAPL'}
  Step 03 | DECISION     |      0 ms | Calling tool 'calculate_financial_metric' with args {'metric': 'pe_ratio', 'price': 0, 'ep
  Step 04 | TOOL_CALL    |      0 ms | get_stock_price({'ticker': 'AAPL'})
  Step 05 | TOOL_CALL    |      0

In [11]:
# Detailed report for Demo 3 (comparison query)
print("\n>>> Full trace for Demo 3 (MSFT vs GOOGL comparison):\n")
obs_logger.print_full_report(trace_id_3)



>>> Full trace for Demo 3 (MSFT vs GOOGL comparison):


#################################################################
  EXECUTION TRACE REPORT — [42e92c33]
#################################################################
  Query        : Compare MSFT and GOOGL: fetch their prices, latest news, then calculate the market cap for each assuming 7,440M shares for MSFT and 12,200M shares for GOOGL.
  Status       : success
  Started      : 2026-06-16T11:57:32.096323
  Finished     : 2026-06-16T11:57:37.654232
  Total time   : 5558 ms
  Est. cost    : $0.000172
  Steps logged : 19

  Step 01 | DECISION     |      0 ms | Calling tool 'get_stock_price' with args {'ticker': 'MSFT'}
  Step 02 | DECISION     |      0 ms | Calling tool 'get_stock_price' with args {'ticker': 'GOOGL'}
  Step 03 | DECISION     |      0 ms | Calling tool 'search_financial_news' with args {'ticker': 'MSFT'}
  Step 04 | DECISION     |      0 ms | Calling tool 'search_financial_news' with args {'ticker': 'GOOGL'}
  

## 8. Observability Dashboard — Cost & Latency

The dashboard aggregates **all traces** in one view — useful for identifying:
- Which queries are slowest (latency hotspots)
- Which runs used the most steps (potential infinite-loop risk)
- Cost trends across different query types


In [12]:
obs_logger.print_summary_dashboard()



  📊  OBSERVABILITY DASHBOARD — ALL TRACES
  Trace    Status   Steps  Time(ms)   Est.Cost    Query
  ------------------------------------------------------------
  536777d3 success       4       2108 $0.000072  What is the current stock price of ...
  0d635cda success      10       3485 $0.000120  Research Apple (AAPL): get its curr...
  42e92c33 success      19       5558 $0.000172  Compare MSFT and GOOGL: fetch their...



### Per-step latency breakdown (Demo 2)

Drill into which **step type** took the most time.


In [13]:
trace = obs_logger.traces[trace_id_2]

step_latencies: Dict[str, List[float]] = {}
for step in trace.steps:
    step_latencies.setdefault(step.step_type, []).append(step.latency_ms)

print(f"\nLatency breakdown for trace [{trace_id_2}]:")
print(f"{'Step Type':15s}  {'Count':6s}  {'Total ms':10s}  {'Avg ms':8s}")
print("-" * 45)
for stype, latencies in sorted(step_latencies.items()):
    total = sum(latencies)
    avg   = total / len(latencies)
    print(f"  {stype:13s}  {len(latencies):6d}  {total:10.1f}  {avg:8.1f}")
print(f"\n  Total wall-clock: {trace.total_ms:.0f} ms")



Latency breakdown for trace [0d635cda]:
Step Type        Count   Total ms    Avg ms  
---------------------------------------------
  decision            3         0.0       0.0
  reasoning           1      2011.9    2011.9
  tool_call           3         0.0       0.0
  tool_result         3       261.7      87.2

  Total wall-clock: 3485 ms


## 9. Exporting Traces (JSON)

In production you would ship traces to an external store (Elastic, BigQuery, S3 …).  
Below we serialise a trace to JSON — ready to ingest into any logging pipeline.


In [14]:
import dataclasses

def trace_to_dict(trace: ExecutionTrace) -> dict:
    """Convert an ExecutionTrace to a plain dict for JSON export."""
    return {
        "trace_id":       trace.trace_id,
        "query":          trace.query,
        "status":         trace.status,
        "started_at":     trace.started_at,
        "finished_at":    trace.finished_at,
        "total_ms":       trace.total_ms,
        "estimated_cost": trace.estimated_cost,
        "final_answer":   trace.final_answer,
        "steps": [
            {
                "step_id":    s.step_id,
                "step_type":  s.step_type,
                "timestamp":  s.timestamp,
                "content":    str(s.content)[:300],
                "latency_ms": s.latency_ms,
                "metadata":   s.metadata,
            }
            for s in trace.steps
        ],
    }

trace_json = json.dumps(trace_to_dict(obs_logger.traces[trace_id_2]), indent=2)
print("Sample JSON export (first 1500 chars):")
print(trace_json[:1500])
print("\n... (truncated for display)")


Sample JSON export (first 1500 chars):
{
  "trace_id": "0d635cda",
  "query": "Research Apple (AAPL): get its current price, find the latest news headlines, then calculate its P/E ratio assuming EPS of $6.57.",
  "status": "success",
  "started_at": "2026-06-16T11:57:28.591725",
  "finished_at": "2026-06-16T11:57:32.076511",
  "total_ms": 3484.786,
  "estimated_cost": 0.00012014999999999999,
  "final_answer": "Here's the latest information on Apple (AAPL):\n\n- **Current Stock Price**: $189.50 (up 1.23% today)\n  \n- **Latest News Headlines**:\n  - Apple reports record iPhone 16 Pro sales in Q1 2025.\n  - Apple Vision Pro 2 launch confirmed for late 2025.\n  - Apple Services revenue grows 18% YoY, analysts bullish.\n\n- **P/E Ratio Calculation**: The P/E ratio is currently calculated as 0.00x, which indicates that the current stock price was not provided for the calculation. If you need the P/E ratio based on the current stock price, please let me know!",
  "steps": [
    {
      "step

## 10. Reflection Questions

1. **Why is observability considered a first-class concern** in production agent systems, not just a debugging afterthought?

2. **How do execution traces differ from traditional application logs?** What extra information do they provide and when does that matter?

3. The `decision` step logs the agent's intent *before* a tool is called, while `tool_result` logs the output *after*. **Why is logging both sides important?** Give a concrete debugging scenario where one side alone would be insufficient.

4. Looking at the latency breakdown in Section 8 — **which step type tends to dominate?** What strategies would you use to reduce end-to-end latency?

5. In this demo cost estimation is very rough (token count × fixed rate). **What additional factors should a production cost tracker capture?**

6. **How would you extend this observability layer to detect anomalies automatically** — for example, a tool call loop or an unexpectedly high step count?

---

**Key Takeaway:**  
Observability transforms an agent from a black box into a transparent system.  
Every decision, tool call, and reasoning step can be captured, inspected, and acted upon — enabling confident debugging, cost control, and compliance auditing in production LangGraph deployments.
